In [ ]:
!pip install pyfaidx

In [ ]:
!mkdir -p data/hg38/
!curl https://storage.googleapis.com/basenji_barnyard2/hg38.ml.fa.gz > data/hg38/hg38.ml.fa.gz
!gunzip data/hg38/hg38.ml.fa.gz
!curl https://storage.googleapis.com/basenji_barnyard2/sequences_human.bed > data/hg38/human-sequences.bed

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  839M  100  839M    0     0  60.7M      0  0:00:13  0:00:13 --:--:-- 84.3M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1112k  100 1112k    0     0  2621k      0 --:--:-- --:--:-- --:--:-- 2618k


In [ ]:
import pandas as pd
pd.read_csv('/content/data/hg38/human-sequences.bed', sep='\t', header=None)

,0,1,2,3
0,chr18,928386,1059458,train
1,chr4,113630947,113762019,train
2,chr11,18427720,18558792,train
3,chr16,85805681,85936753,train
4,chr3,158386188,158517260,train
...,...,...,...,...
38166,chr19,33204702,33335774,test
38167,chr14,41861379,41992451,test
38168,chr19,30681544,30812616,test
38169,chr14,61473198,61604270,test


In [ ]:
import pandas as pd
from pyfaidx import Fasta
import os

FASTA_PATH = "./data/hg38/hg38.ml.fa"
INTERVALS_TSV = "./data/hg38/human-sequences.bed"
OUTPUT_DIR = "."
WINDOW_SIZE = 512
STRIDE = 512
MAX_N_FRACTION = 0

genome = Fasta(FASTA_PATH)


In [ ]:
intervals = pd.read_csv(INTERVALS_TSV, sep="\t", names=[ 'chrom', 'start', 'end', 'split'])

In [ ]:
out_files = {
    split: open(os.path.join(OUTPUT_DIR, f"{split}.txt"), "w")
    for split in intervals["split"].unique()
}

for _, row in intervals.iterrows():
    chrom, start, end, split = row["chrom"], int(row["start"]), int(row["end"]), row["split"]

    if chrom not in genome:
        continue

    chrom_len = len(genome[chrom])
    end = min(end, chrom_len)

    for win_start in range(start, end - WINDOW_SIZE + 1, STRIDE):
        win_end = win_start + WINDOW_SIZE
        seq = str(genome[chrom][win_start:win_end]).upper()

        if seq.count("N") / WINDOW_SIZE > MAX_N_FRACTION:
            continue

        out_files[split].write(seq + "\n")

for f in out_files.values():
    f.close()

In [ ]:
!wc -l train.txt

8709271 train.txt


In [ ]:
!head -n 250000 train.txt > train_250k.txt
!cp train_250k.txt drive/MyDrive/nlp2dna/

!head -n 5000 valid.txt > valid_5k.txt
!cp valid_5k.txt drive/MyDrive/nlp2dna/

In [ ]:
!cp test.txt drive/MyDrive/nlp2dna/
!cp valid.txt drive/MyDrive/nlp2dna/
!cp train.txt drive/MyDrive/nlp2dna/